# Reconstruction Example Using lftomo

Environment Requirements:

See x

In [17]:
from multiprocessing import shared_memory, Pool
from multiprocessing.managers import SharedMemoryManager
import os
from pathlib import Path
import sys

# Image Display
import cv2
from ipywidgets import Image
# Correct ordering if input not zero-padded
from natsort import natsorted
# Used for everything
import numpy as np
# Used to get processor data for setting multiprocessing.
import psutil
# Default file format is tif
import tifffile as tf
# Used for Recon
import tomopy

sys.path.insert(0, os.path.abspath('..'))

# Flexible logging to screen and file, in a parseable fixed-width format.
from lftomo.util.log import log
# Windowing to select only parts of an image at a time.
from lftomo.util.window import DataWindowSet, ProjOrder, ReconOrder, SinoOrder, get_tilt_pad, CTFov, ImgShape
# Handling flatfield backgrounds.
from lftomo.prep.flat import FLAT, weighted_normalize, calculate_shift
# Shared Memory Handling
from lftomo.util.mem import SharedNP
# Reading and Writing files distributed
from lftomo.util.io import distribute_read, write_slice, np_convert, write_centers
# Stripe removal
from lftomo.prep.stripe import remove_sort_vo, remove_large_vo, remove_dead_vo
# Threshold normalization
# from lftomo.prep.norm import normalize
# Rotation and Sample Movement Alignment
# from lftomo.prep.align import rotate_xy

# Phase Retrieval
from lftomo.prep.phase import phase_retrieval

## Setup

### Reconstruction Parameters

Starting with basics (input path, window size, output location)

In [18]:
log.set_logdir(Path("..", "data", "logs"))
log.header()
# smm = SharedMemoryManager()
# smm.start()

TYPE  |STEP                |   TIMESTAMP   |MEM USAGE|MEM FREE | STATEMENT 


In [19]:

window_size = psutil.cpu_count()
input_path = Path("..", "data", "input", "Sam19")
flat_path = Path("..", "data", "input", "Sam19_Flats")
output_base = Path("..", "data", "output")

# Field of View Angles.
fov = CTFov(Theta=None, Degrees=180, Count=1)

internal_dtype = np.float32		# Data type to use internally. 
cf_count = 8 					# Number of centerfind sections to generate.
proc_count = 12					# Number of simulataneous processes.
center_count = 20				# Number of centers to generate.

In [20]:
data_set = DataWindowSet.from_folder(input_path, ProjOrder, window_size, fov)
input_files = natsorted(Path(input_path).glob("**/*.tif*"))

### Flat Field Generation

In [21]:
flat_mem = SharedNP(f'flats_ipynb', internal_dtype, data_set.data_shape._replace(Theta=len(FLAT))).create()

with flat_mem as flat_set:
	for flat in list(FLAT):
		flat_set[flat.index, :, :] = tf.imread(flat_path.joinpath(f"{flat}_median.tiff")).astype(internal_dtype)
	
	# Precalculation of pregain and postgain with darks.
	flat_set[FLAT.PREGAIN.index] -= flat_set[FLAT.PREDARK.index]
	flat_set[FLAT.POSTGAIN.index] -= flat_set[FLAT.POSTDARK.index]


INFO  |Shared Memory       |01:02:55.846236|0003.00GB|0041.00GB|"Shared Memory Name flats_ipynb Pre-Existed"
INFO  |Shared Memory       |01:02:55.848882|0003.00GB|0041.00GB|"Shared Memory Name flats_ipynb Recreated"


In [22]:
with flat_mem as flat_set:
	flat_intermediate = output_base.joinpath("flat_intermediate")
	flat_intermediate.mkdir(parents=True, exist_ok=True)
	tf.imwrite(Path(flat_intermediate, "pregain.tiff"), flat_set[FLAT.PREGAIN.index])
	tf.imwrite(Path(flat_intermediate, "postgain.tiff"), flat_set[FLAT.POSTGAIN.index])
	tf.imwrite(Path(flat_intermediate, "predark.tiff"), flat_set[FLAT.PREDARK.index])
	tf.imwrite(Path(flat_intermediate, "postdark.tiff"), flat_set[FLAT.POSTDARK.index])

	if False:
		# Encode into in-memory PNG for Jupyter
		# _, pregain = cv2.imencode('.png', flat_set[FLAT.PREGAIN.index])
		# _, postgain = cv2.imencode('.png', flat_set[FLAT.POSTGAIN.index])

		# This 'works' but requires leveling
		Image(value=pregain, width=data_set.data_shape.X, height=data_set.data_shape.Y)

### Angle Calculation

In [23]:
# No angle dropping so skipping for now.

## Center Finding

Steps:

1. Read Slice(s)
2. Data Cleanup
3. Center Calculation
4. Angle Calculation
5. Reconstruct & Write Slices


### Read Data

In [24]:
center_shape = SinoOrder(**data_set.data_shape._asdict())._replace(Y=cf_count)
cf_read_mem = SharedNP(f"cf_read_ipynb", data_set.dtype, center_shape).create()

cf_dataset = data_set.at_interval(cf_count)
vo_centers = {}
pc_centers = {}
man_centers = {}

for i, cf_data in enumerate(cf_dataset):
	log.write("Center Finding", f"{cf_data}, {cf_read_mem.buffer_address}")
	distribute_read(cf_read_mem[i], cf_data, input_files, sino_order=True)
	with cf_read_mem as cf_slice:
		log.write("CF Test", f"{cf_slice.shape}")
		vo_centers[cf_data.start] = tomopy.find_center_vo(cf_slice, i)
		

log.write("Center Finds", f"Vo CFs: {vo_centers}")

INFO  |Shared Memory       |01:02:56.089263|0003.00GB|0041.00GB|"Shared Memory Name cf_read_ipynb Pre-Existed"
INFO  |Shared Memory       |01:02:56.090858|0003.00GB|0041.00GB|"Shared Memory Name cf_read_ipynb Recreated"
TIME  |Center Finding      |01:02:56.092845|0003.00GB|0041.00GB|"ProjOrder(Theta=1501, Y=1, X=2048) (uint16) at 0, slice(None, None, None)"
TIME  |Mem Read            |01:02:56.094731|0003.00GB|0041.00GB|"0|1"
TIME  |CF Test             |01:02:56.236829|0003.00GB|0041.00GB|"(8, 1501, 2048)"
TIME  |Center Finding      |01:02:57.336212|0003.00GB|0041.00GB|"ProjOrder(Theta=1501, Y=1, X=2048) (uint16) at 256, slice(None, None, None)"
TIME  |Mem Read            |01:02:57.339254|0003.00GB|0041.00GB|"6148096|1"
TIME  |CF Test             |01:02:57.472180|0003.00GB|0041.00GB|"(8, 1501, 2048)"
TIME  |Center Finding      |01:02:58.054381|0003.00GB|0041.00GB|"ProjOrder(Theta=1501, Y=1, X=2048) (uint16) at 512, slice(None, None, None)"
TIME  |Mem Read            |01:02:58.057757|00

### Data Cleanup

Flatfield correction and stripe removal.

In [25]:
cf_mem = SharedNP(f"cf_ipynp", internal_dtype, SinoOrder(**data_set.data_shape._asdict())._replace(Y=cf_count)).create()
cf_win = cf_dataset.full_window()

with Pool(proc_count) as pool:
	pool.starmap(weighted_normalize, [(cf_mem, flat_mem, cf_read_mem, cf_win, i, True) for i in range(cf_mem.shape.Theta)])
	log.write("Normalize", f"{cf_win.internal.Y()}|{cf_win.Y()}")

remove_dead_vo(cf_win, cf_mem)
log.write("Stripe Removal (D,L)", f"{cf_win.internal.Y()}|{cf_win.Y()}")
remove_sort_vo(cf_win, cf_mem)
log.write("Stripe Removal (s)", f"{cf_win.internal.Y()}|{cf_win.Y()}")


INFO  |Shared Memory       |01:03:01.483098|0003.00GB|0041.00GB|"Shared Memory Name cf_ipynp Pre-Existed"
INFO  |Shared Memory       |01:03:01.484372|0003.00GB|0041.00GB|"Shared Memory Name cf_ipynp Recreated"
TIME  |Normalize           |01:03:01.876739|0003.00GB|0040.00GB|"slice(0, 8, 1)|slice(0, 2048, 256)"
TIME  |Stripe Removal (D,L)|01:03:06.374781|0003.00GB|0041.00GB|"slice(0, 8, 1)|slice(0, 2048, 256)"
TIME  |Stripe Removal (s)  |01:03:07.665286|0003.00GB|0041.00GB|"slice(0, 8, 1)|slice(0, 2048, 256)"


### Center Calculation

Here, calculation by find_center_pc, then writing centers around that for manual identification.

In [26]:
with cf_mem as cf_slice:
	for i, cf_data in enumerate(cf_dataset):
		vo_centers[cf_data.start] = tomopy.find_center_vo(cf_slice, i)

log.write("Center Find (VO)", f"Vo CFs: {vo_centers}")

pc_centers = tomopy.find_center_pc(tf.imread(input_files[0]), tf.imread(input_files[1]))
log.write("Center Find (PC)", f"Tomopy Center {pc_centers}")
base_center = 1036.0
tilt = 0

TIME  |Center Find (VO)    |01:03:10.838612|0003.00GB|0041.00GB|"Vo CFs: {0: 1080.0, 256: 1080.0, 512: 1080.0, 768: 1080.0, 1024: 1080.0, 1280: 1080.0, 1536: 1080.0, 1792: 1080.0}"
TIME  |Center Find (PC)    |01:03:12.335911|0003.00GB|0041.00GB|"Tomopy Center 1023.5"


### Reconstruct & Write Slices

In [27]:

for i, cf_data in enumerate(cf_dataset):
	log.write("Center Writing", f"{len(data_set.angles)}_{cf_mem.shape})")
	center_path = output_base.joinpath("cf_data", f"s{str(cf_data.start).zfill(5)}")
	center_path.mkdir(parents=True, exist_ok=True)
	write_centers(cf_mem[i], center_path, np.linspace(base_center - 5.0, base_center + 5.0, center_count), data_set.angles, calc_entropy=False)

	center_path = output_base.joinpath("cf_data", f"noNorm_s{str(cf_data.start).zfill(5)}")
	center_path.mkdir(parents=True, exist_ok=True)
	write_centers(cf_read_mem[i], center_path, np.linspace(base_center - 5.0, base_center + 5.0, center_count), data_set.angles, calc_entropy=False)

TIME  |Center Writing      |01:03:12.354906|0003.00GB|0041.00GB|"1501_SinoOrder(Y=8, Theta=1501, X=2048))"
INFO  |Shared Memory       |01:03:12.361220|0003.00GB|0041.00GB|"Shared Memory Name cf_write_mem Pre-Existed"
INFO  |Shared Memory       |01:03:12.408889|0004.00GB|0041.00GB|"Shared Memory Name cf_write_mem Recreated"
INFO  |Shared Memory       |01:03:18.330476|0003.00GB|0041.00GB|"Unlinking cf_write_mem"
STATUS|Shared Memory       |01:03:18.331610|0003.00GB|0041.00GB|"Shared Memory cf_write_mem Is None"
INFO  |Shared Memory       |01:03:18.334010|0003.00GB|0041.00GB|"Shared Memory Name cf_write_mem Pre-Existed"
INFO  |Shared Memory       |01:03:18.367091|0003.00GB|0041.00GB|"Shared Memory Name cf_write_mem Recreated"
INFO  |Shared Memory       |01:03:23.117473|0003.00GB|0041.00GB|"Unlinking cf_write_mem"
STATUS|Shared Memory       |01:03:23.118528|0003.00GB|0041.00GB|"Shared Memory cf_write_mem Is None"
TIME  |Center Writing      |01:03:23.120123|0003.00GB|0041.00GB|"1501_SinoOrd

In [28]:
cf_read_mem.close()
cf_read_mem.unlink()

STATUS|Shared Memory       |01:04:35.916771|0003.00GB|0041.00GB|"Shared Memory cf_read_ipynb Unlinked"


In [29]:
cf_mem.close()
cf_mem.unlink()

STATUS|Shared Memory       |01:04:35.947092|0003.00GB|0041.00GB|"Shared Memory cf_ipynp Unlinked"


## Reconstruction

Loop, looking at small windows each time.

1. Read Data
2. Clean Data (Pre-Rotation Steps)
3. Rotate Data
4. Clean Data (Post-Rotation Steps)
5. Reconstruct Slices
6. Normalize Slices

In [34]:
phase_alpha = 0.2

with (SharedNP(f"sino_ipynb", internal_dtype, SinoOrder(**data_set.window_shape._asdict())) as sino_mem,
	SharedNP(f"workread_ipynb", data_set.dtype, SinoOrder(**data_set.window_shape._asdict())) as workmem_read,
	SharedNP(f"recon_ipynb", internal_dtype, ReconOrder(Y=data_set.window_size, Z=data_set.window_shape.X, X=data_set.window_shape.X)) as recon_mem):

	for j, dw in enumerate(data_set):
		log.write("Reading Data", f"{data_set.window_shape} : {dw} : {workmem_read.shape} : {sino_mem.shape}")
		distribute_read(workmem_read[dw.internal.Y()], dw, input_files, sino_order=True)
		log.write("Data Read")

		with Pool(proc_count) as pool:
			log.write("Normalize Prep", f"{dw.internal.Y()}|{dw.Y()}")
			pool.starmap(weighted_normalize, [(sino_mem, flat_mem, workmem_read, dw, i, True) for i in range(sino_mem.shape.Theta)])
			log.write("Flat Field Adjustment", f"{dw.internal.Y()}|{dw.Y()}")
		remove_dead_vo(dw, sino_mem)
		log.write("Stripe Removal (D,L)", f"{dw.internal.Y()}|{dw.Y()}")
		remove_sort_vo(dw, sino_mem)
		log.write("Stripe Removal (S)", f"{dw.internal.Y()}|{dw.Y()}")

		with SharedNP(f"workmem_rot_ipynb", internal_dtype, data_set.window_shape) as workmem_rot:
			if tilt != 0:
				# rotate_xy(dw, sino_mem, workmem_rot, base_center, tilt)
				pass
			else:
				with sino_mem as sino, workmem_rot as target_data:
					target_data[:, :, :] = np.transpose(sino[:, :, :], [1, 0, 2])
					log.write("Rotation", f"Data Copy from {sino.shape} to {target_data.shape}")

			phase_retrieval(dw, workmem_rot, 0.006, 0.743 / 10000, 13.8, 3, False, False)

			with workmem_rot as input_data, recon_mem as output_data:
				output_data[:, :, :] = tomopy.recon(input_data[:, :, :], data_set.angles, sinogram_order=False, center=base_center, algorithm="gridrec", ncore=proc_count)	

			with Pool(proc_count) as pool:
				output_base.joinpath("recon").mkdir(exist_ok=True, parents=True)
				log.write("Writing Recon", f"{dw.Y()}")

				pool.starmap(write_slice, [(recon_mem[0:data_set.window_size], output_base.joinpath("recon", f"{str(i).zfill(5)}.tif"), i - dw.Y().start)
												for i in dw.y_range])


INFO  |Shared Memory       |01:52:34.506056|0003.00GB|0040.00GB|"Shared Memory Name sino_ipynb Pre-Existed"
INFO  |Shared Memory       |01:52:34.527896|0003.00GB|0040.00GB|"Shared Memory Name sino_ipynb Recreated"
INFO  |Shared Memory       |01:52:34.529064|0003.00GB|0040.00GB|"Shared Memory Name workread_ipynb Pre-Existed"
INFO  |Shared Memory       |01:52:34.540437|0003.00GB|0041.00GB|"Shared Memory Name workread_ipynb Recreated"
INFO  |Shared Memory       |01:52:34.541961|0003.00GB|0041.00GB|"Shared Memory Name recon_ipynb Pre-Existed"
INFO  |Shared Memory       |01:52:34.566060|0003.00GB|0041.00GB|"Shared Memory Name recon_ipynb Recreated"
TIME  |Reading Data        |01:52:34.568125|0003.00GB|0041.00GB|"ProjOrder(Theta=1501, Y=12, X=2048) : ProjOrder(Theta=1501, Y=12, X=2048) (uint16) at 0 : SinoOrder(Y=12, Theta=1501, X=2048) : SinoOrder(Y=12, Theta=1501, X=2048)"
TIME  |Mem Read            |01:52:34.571060|0003.00GB|0041.00GB|"0|12"
TIME  |Data Read           |01:52:34.764985|000

In [35]:
flat_mem.close()
flat_mem.unlink()

STATUS|Shared Memory       |02:17:57.718576|0003.00GB|0041.00GB|"Shared Memory flats_ipynb Unlinked"


In [36]:
sino_mem.close()
sino_mem.unlink()
workmem_read.close()
workmem_read.unlink()
recon_mem.close()
recon_mem.unlink()

STATUS|Shared Memory       |02:17:57.762829|0003.00GB|0041.00GB|"Shared Memory sino_ipynb Unlinked"


STATUS|Shared Memory       |02:17:57.764216|0003.00GB|0041.00GB|"Shared Memory workread_ipynb Is None"
STATUS|Shared Memory       |02:17:57.783721|0003.00GB|0041.00GB|"Shared Memory recon_ipynb Unlinked"
